# Soldani — Multimodal Causal Fairness Pipeline

Combina causal discovery, analisi intersezionale, multi-model benchmark e report visuale.


## 1. Setup del path e import librerie

Configura il path di progetto e importa tutte le librerie necessarie:
FairMind (src.*), i moduli custom (src_soldani.*), pgmpy per struttura, pandas, plotly.


In [27]:
from pathlib import Path
import sys
current = Path.cwd()
while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [28]:
import json
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import time

from src.graph import build_sfm
from src_soldani import discover_graph, learn_sfm, graph_similarity
from src_soldani import combine_sensitive_attrs, compute_intersectional_effects
from src_soldani import generate_html_report
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect, spurious_effect,
    natural_direct_effect, natural_indirect_effect,
)
from src.visualisation.graph import visualize_sfm

from pgmpy.estimators import BayesianEstimator


In [29]:
# Config LLM per THOR: legge LLAMA_HOST dall'ambiente (fallback localhost)
import os
from src.llm import LLM_CONFIGS

LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
LLM_CONFIGS[0]["base_url"] = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1"
print(f"LLM endpoint configurato: http://{LLAMA_HOST}:{LLAMA_PORT}/v1")


LLM endpoint configurato: http://localhost:8080/v1


## 2. Configurazione del benchmark

Definisce il dataset Adult, gli attributi protetti, mediatori, confounders
e le variabili per il causal discovery.


In [30]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col": "T_income",
    "target_val": ">50K",
    "protected": "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "sensitive_attrs_intersectional": ["S2_gender", "S3_race"],
    "mediators": ["hours-per-week"],
    "confounders": ["education"],
    "all_vars_for_discovery": [
        "S2_gender", "S3_race", "education", "hours-per-week",
        "occupation", "marital-status", "T_income"
    ],
}


## 3. Fase 1 — Causal Discovery

Invece di specificare manualmente il grafo, usiamo HillClimbSearch (pgmpy)
per imparare la struttura causale dai dati. Poi confrontiamo il grafo scoperto
con lo SFM manuale usando metriche strutturali (SHD, precision, recall, F1).


In [31]:
print("=" * 60)
print("FASE 1: Causal Discovery")
print("=" * 60)

df = pd.read_csv(CONFIG["csv_path"])

# Seleziona variabili per discovery
discovery_cols = CONFIG["all_vars_for_discovery"]
df_disc = df[discovery_cols].dropna().copy()

# Discretizza hours-per-week
df_disc["hours-per-week"] = pd.cut(
    df_disc["hours-per-week"], bins=[0, 20, 35, 45, 60, 100],
    labels=["<=20", "21-35", "36-45", "46-60", ">60"], include_lowest=True
)

print(f"Dataset per discovery: {df_disc.shape[0]} righe, {df_disc.shape[1]} colonne")
print(f"Variabili: {list(df_disc.columns)}")


FASE 1: Causal Discovery
Dataset per discovery: 48842 righe, 7 colonne
Variabili: ['S2_gender', 'S3_race', 'education', 'hours-per-week', 'occupation', 'marital-status', 'T_income']


In [32]:
start_disc = time.perf_counter()

learned = learn_sfm(
    data=df_disc,
    outcome_attr=CONFIG["target_col"],
    score="bdeu",
    max_indegree=4,
)

discovery_time = time.perf_counter() - start_disc
print(f"Discovery completata in {discovery_time:.4f}s")
print(f"Numero nodi: {learned.number_of_nodes()}")
print(f"Numero archi: {learned.number_of_edges()}")
print(f"Archi: {list(learned.edges())}")


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'S2_gender': 'C', 'S3_race': 'C', 'education': 'C', 'hours-per-week': 'O', 'occupation': 'C', 'marital-status': 'C', 'T_income': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'S2_gender': 'C', 'S3_race': 'C', 'education': 'C', 'hours-per-week': 'O', 'occupation': 'C', 'marital-status': 'C', 'T_income': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'S2_gender': 'C', 'S3_race': 'C', 'education': 'C', 'hours-per-week': 'O', 'occupation': 'C', 'marital-status': 'C', 'T_income': 'C'}


Discovery completata in 0.1967s
Numero nodi: 7
Numero archi: 10
Archi: [('S2_gender', 'marital-status'), ('S2_gender', 'S3_race'), ('education', 'occupation'), ('education', 'T_income'), ('hours-per-week', 'marital-status'), ('hours-per-week', 'S2_gender'), ('occupation', 'S2_gender'), ('occupation', 'hours-per-week'), ('marital-status', 'T_income'), ('marital-status', 'S3_race')]


In [33]:
manual_sfm = build_sfm(
    sensitive_attr=CONFIG["protected"],
    outcome_attr=CONFIG["target_col"],
    confounder_attrs=CONFIG["confounders"],
    mediator_attrs=CONFIG["mediators"],
)

common_nodes = list(set(learned.nodes()) & set(manual_sfm.nodes()))
sim = graph_similarity(learned, manual_sfm, nodes=common_nodes)
print("Similarità strutturale tra grafo scoperto e SFM manuale:")
for k, v in sim.items():
    print(f"  {k}: {v}")


Similarità strutturale tra grafo scoperto e SFM manuale:
  shd: 6
  precision: 0.1667
  recall: 0.5
  f1: 0.25
  jaccard: 0.1429


## 4. Fase 2 — FairMind: Ground Truth su grafo manuale

Calcola i 5 effetti causali sul grafo SFM specificato dall'esperto.
Questi valori sono il riferimento (ground truth) per i confronti successivi.


In [ ]:
print("\n" + "=" * 60)
print("FASE 2: FairMind — Ground Truth (grafo manuale)")
print("=" * 60)

def run_fairmind(config):
    df = pd.read_csv(config["csv_path"])
    cols = [config["protected"]] + config["mediators"] + config["confounders"] + [config["target_col"]]
    df = df[cols].dropna()

    # Stesso binning usato in FASE 1 (discovery) e in build_llm_prompt: senza
    # questo, il BN viene fittato su hours-per-week grezzo (99 valori distinti)
    # invece delle 5 fasce, rendendo gli effetti non confrontabili con LLM/discovery.
    if "hours-per-week" in df.columns:
        df["hours-per-week"] = pd.cut(
            df["hours-per-week"], bins=[0, 20, 35, 45, 60, 100],
            labels=["<=20", "21-35", "36-45", "46-60", ">60"], include_lowest=True
        )

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=config["mediators"],
    )
    bn = fit_discrete_bayesian_model(
        sfm=sfm, data=df,
        estimator_instance=(BayesianEstimator, {"prior_type": "BDeu"}),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    effects = {
        "TV": total_variation(bn, target, config["protected"], x0, x1),
        "TE": total_effect(bn, target, config["protected"], x0, x1),
        "SE": spurious_effect(bn, target, config["protected"], x0),
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start
    return effects, elapsed

manual_effects, fm_time = run_fairmind(CONFIG)
print(f"FairMind completato in {fm_time:.4f}s")
for k, v in manual_effects.items():
    print(f"  {k}: {v:.6f}")

## 5. Fase 3 — FairMind su grafo scoperto

Ripete il calcolo degli effetti usando il grafo appreso via causal discovery.
Il confronto con il grafo manuale mostra quanto la struttura influisca
sui risultati finali.


In [ ]:
print("\n" + "=" * 60)
print("FASE 3: FairMind — Effetti su grafo scoperto")
print("=" * 60)

def run_fairmind_on_graph(graph, config):
    df = pd.read_csv(config["csv_path"])
    graph_nodes = set(graph.nodes())
    cols = [c for c in [config["protected"]] + config["mediators"] + config["confounders"] + [config["target_col"]] if c in graph_nodes]
    extra = [c for c in graph_nodes if c in df.columns and c not in cols]
    cols = cols + extra
    df = df[cols].dropna()

    # Il grafo e' stato imparato su df_disc (FASE 1), dove hours-per-week e'
    # gia' binnato in 5 fasce: va rifatto qui identico, altrimenti il BN
    # viene fittato su hours-per-week grezzo (99 stati) mentre la struttura
    # del grafo assume 5 stati.
    if "hours-per-week" in df.columns:
        df["hours-per-week"] = pd.cut(
            df["hours-per-week"], bins=[0, 20, 35, 45, 60, 100],
            labels=["<=20", "21-35", "36-45", "46-60", ">60"], include_lowest=True
        )

    bn = fit_discrete_bayesian_model(
        sfm=graph, data=df,
        estimator_instance=(BayesianEstimator, {"prior_type": "BDeu"}),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    effects = {
        "TV": total_variation(bn, target, config["protected"], x0, x1),
        "TE": total_effect(bn, target, config["protected"], x0, x1),
        "SE": spurious_effect(bn, target, config["protected"], x0),
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start
    return effects, elapsed

learned_effects, learned_time = run_fairmind_on_graph(learned, CONFIG)
print(f"FairMind su grafo scoperto completato in {learned_time:.4f}s")
for k, v in learned_effects.items():
    print(f"  {k}: {v:.6f}")

## 6. Fase 4 — Analisi Intersezionale

Crea un attributo protetto composito (es. Genere × Razza) e calcola gli effetti
causali per ogni coppia di gruppi intersezionali. La heatmap mostra quali
combinazioni subiscono la disparità più alta.


In [36]:
print("\n" + "=" * 60)
print("FASE 4: Analisi Intersezionale")
print("=" * 60)

df_inter = pd.read_csv(CONFIG["csv_path"])
df_inter = combine_sensitive_attrs(df_inter, CONFIG["sensitive_attrs_intersectional"])

print("Gruppi intersezionali creati:")
print(df_inter["_sensitive_composite"].value_counts().to_string())



FASE 4: Analisi Intersezionale
Gruppi intersezionali creati:
_sensitive_composite
Male×White                   28735
Female×White                 13027
Male×Black                    2377
Female×Black                  2308
Male×Asian-Pac-Islander       1002
Female×Asian-Pac-Islander      517
Male×Amer-Indian-Eskimo        285
Male×Other                     251
Female×Amer-Indian-Eskimo      185
Female×Other                   155


In [37]:
start_inter = time.perf_counter()

inter_effects = compute_intersectional_effects(
    df=pd.read_csv(CONFIG["csv_path"]),
    sensitive_attrs=CONFIG["sensitive_attrs_intersectional"],
    target_col=CONFIG["target_col"],
    target_val=CONFIG["target_val"],
    mediators=CONFIG["mediators"],
    confounders=CONFIG["confounders"],
)

inter_time = time.perf_counter() - start_inter
print(f"Analisi intersezionale completata in {inter_time:.4f}s")
print("\nEffetti per coppia intersezionale (prime 20):")
for i, (pair, effs) in enumerate(sorted(inter_effects.items())):
    if i >= 20:
        print(f"  ... e altre {len(inter_effects) - 20} coppie")
        break
    print(f"  {pair}: TV={effs['TV']:.4f}, TE={effs['TE']:.4f}")


2026-07-14 09:47:46.487 | DEBUG    | src.model:fit_discrete_bayesian_model:33 - Using estimator: <class 'pgmpy.estimators.BayesianEstimator.BayesianEstimator'> with parameters: {'prior_type': 'BDeu'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'_sensitive_composite': 'C', 'hours-per-week': 'N', 'education': 'C', 'T_income': 'C'}
2026-07-14 09:47:46.948 | DEBUG    | src.effects:total_variation:248 - Computing total variation for target=('T_income', '>50K'), private_baseline=Female×Amer-Indian-Eskimo, private_mod=Female×Asian-Pac-Islander
2026-07-14 09:47:46.952 | DEBUG    | src.effects:spurious_effect:195 - Computing spurious effect for target=('T_income', '>50K'), private_value=Female×Amer-Indian-Eskimo
2026-07-14 09:47:46.957 | DEBUG    | src.model:fit_discrete_bayesian_model:33 - Using estimator: <class 'pgmpy.estimators.BayesianEstimator.BayesianEstimator'> with parameters: {'prior_type': 'BDeu'}
INFO:pgmpy: Datatype (N=n

Analisi intersezionale completata in 20.7385s

Effetti per coppia intersezionale (prime 20):
  (Female×Amer-Indian-Eskimo, Female×Asian-Pac-Islander): TV=0.0516, TE=-0.0102
  (Female×Amer-Indian-Eskimo, Female×Black): TV=-0.0249, TE=-0.0472
  (Female×Amer-Indian-Eskimo, Female×Other): TV=-0.0099, TE=-0.0279
  (Female×Amer-Indian-Eskimo, Female×White): TV=0.0362, TE=-0.0049
  (Female×Amer-Indian-Eskimo, Male×Amer-Indian-Eskimo): TV=0.0588, TE=0.0701
  (Female×Amer-Indian-Eskimo, Male×Asian-Pac-Islander): TV=0.2572, TE=0.1329
  (Female×Amer-Indian-Eskimo, Male×Black): TV=0.1004, TE=0.0964
  (Female×Amer-Indian-Eskimo, Male×Other): TV=0.0739, TE=0.0674
  (Female×Amer-Indian-Eskimo, Male×White): TV=0.2333, TE=0.1812
  (Female×Asian-Pac-Islander, Female×Amer-Indian-Eskimo): TV=-0.0516, TE=0.0102
  (Female×Asian-Pac-Islander, Female×Black): TV=-0.0765, TE=-0.0370
  (Female×Asian-Pac-Islander, Female×Other): TV=-0.0615, TE=-0.0177
  (Female×Asian-Pac-Islander, Female×White): TV=-0.0154, TE=0.

In [38]:
# Heatmap TV per coppia intersezionale
pairs = list(inter_effects.keys())
tv_values = [inter_effects[p]['TV'] for p in pairs]

fig = go.Figure(data=go.Heatmap(
    z=[tv_values],
    y=["TV"],
    x=pairs,
    colorscale="RdBu_r",
    zmid=0,
    text=[[f"{v:.4f}" for v in tv_values]],
    texttemplate="%{text}",
    textfont={"size": 10},
))
fig.update_layout(title="Total Variation per coppia intersezionale",
                  xaxis_tickangle=45, height=300, margin=dict(b=150))
fig.show()


## 7. Fase 5 — Costruzione prompt LLM

Costruisce il prompt con le tabelle pre-aggregate per il benchmark multi-modello.


In [39]:
print("\n" + "=" * 60)
print("FASE 5a: Costruzione prompt LLM")
print("=" * 60)

def build_llm_prompt(config):
    df = pd.read_csv(config["csv_path"])
    cols = [config["protected"]] + config["mediators"] + config["confounders"] + [config["target_col"]]
    df = df[cols].dropna()

    protected = config['protected']
    target = config['target_col']
    target_val = config['target_val']
    confounders = config['confounders']

    if "hours-per-week" in config["mediators"]:
        df["hours-per-week_bin"] = pd.cut(
            df["hours-per-week"], bins=[0, 20, 35, 45, 60, 100],
            labels=["<=20", "21-35", "36-45", "46-60", ">60"],
            include_lowest=True,
        )
        mediators = ["hours-per-week_bin"]
    else:
        mediators = config['mediators']

    df["_y"] = (df[target] == target_val).astype(int)

    p_y_given_x = df.groupby(protected, observed=True)['_y'].mean().round(4).reset_index()
    p_y_given_x = p_y_given_x.rename(columns={'_y': 'P(Y=y|X)'})

    p_z = df.groupby(confounders, observed=True).size().reset_index(name='count')
    p_z['P(Z)'] = (p_z['count'] / len(df)).round(4)
    p_z = p_z.drop(columns='count')

    p_y_given_xz = df.groupby([protected] + confounders, observed=True)['_y'].mean().round(4).reset_index()
    p_y_given_xz = p_y_given_xz.rename(columns={'_y': 'P(Y=y|X,Z)'})

    p_w_given_xz = df.groupby([protected] + confounders + mediators, observed=True).size().reset_index(name='count')
    gt = p_w_given_xz.groupby([protected] + confounders, observed=True)['count'].transform('sum')
    p_w_given_xz['P(W|X,Z)'] = (p_w_given_xz['count'] / gt).round(4)
    p_w_given_xz = p_w_given_xz.drop(columns='count')

    p_y_given_xwz = df.groupby([protected] + mediators + confounders, observed=True)['_y'].mean().round(4).reset_index()
    p_y_given_xwz = p_y_given_xwz.rename(columns={'_y': 'P(Y=y|X,W,Z)'})

    def to_csv(d):
        return d.to_csv(index=False)

    prompt = '''You are a causal fairness expert. Compute five causal fairness effects'''
    prompt += ''' using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).'''
    prompt += '''\n\nYou are given PRE-AGGREGATED CONDITIONAL PROBABILITY TABLES'''
    prompt += f''' computed from the dataset (n={len(df)} rows).'''
    prompt += '\nNote: "hours-per-week" has been discretized into bins: <=20, 21-35, 36-45, 46-60, >60.'
    prompt += f'''\n\nVARIABLE ROLES:\n- X (protected): "{protected}", x0="{config['x0']}", x1="{config['x1']}"'''
    prompt += f'''\n- Y (target): "{target}", target state="{target_val}"'''
    prompt += f'''\\n- W (mediators): {mediators}'''
    prompt += f'''\n- Z (confounders): {confounders}'''
    prompt += '''\n\nTABLE 1 \u2014 P(Y=y | X):\n''' + to_csv(p_y_given_x)
    prompt += '''\n\nTABLE 2 \u2014 P(Z):\n''' + to_csv(p_z)
    prompt += '''\n\nTABLE 3 \u2014 P(Y=y | X, Z):\n''' + to_csv(p_y_given_xz)
    prompt += '''\n\nTABLE 4 \u2014 P(W | X, Z):\n''' + to_csv(p_w_given_xz)
    prompt += '''\n\nTABLE 5 \u2014 P(Y=y | X, W, Z):\n''' + to_csv(p_y_given_xwz)
    prompt += '''\n\nIDENTIFICATION FORMULAE:\n'''
    prompt += '''\n- TV = P(Y=y | X=x1) - P(Y=y | X=x0)'''
    prompt += '''\n- TE = sum_z [P(Y=y|x1,z) - P(Y=y|x0,z)] * P(z)'''
    prompt += '''\n- SE = TV - TE'''
    prompt += '''\n- DE = sum_z,w [P(Y=y|x1,w,z) - P(Y=y|x0,w,z)] * P(w|x0,z) * P(z)'''
    prompt += '''\n- IE = sum_z,w P(Y=y|x0,w,z) * [P(w|x1,z) - P(w|x0,z)] * P(z)'''
    prompt += '''\n\nEnd with FINAL_JSON:{"TV":<float>,"TE":<float>,"SE":<float>,"DE":<float>,"IE":<float>}'''
    return prompt

prompt = build_llm_prompt(CONFIG)
print(f"Prompt costruito: {len(prompt)} caratteri, ~{len(prompt) // 4} token stimati")



FASE 5a: Costruzione prompt LLM
Prompt costruito: 10439 caratteri, ~2609 token stimati


## 8. Fase 6 — Multi-model Benchmark

Invia il prompt a TUTTI i modelli LLM registrati e confronta i risultati.


In [40]:
print("\n" + "=" * 60)
print("FASE 6: Multi-model Benchmark")
print("=" * 60)

from src.llm import LLM_CONFIGS, benchmark_models

print("Avvio benchmark multi-modello...")
start_bench = time.perf_counter()
llm_results = benchmark_models(prompt, configs=LLM_CONFIGS)
bench_time = time.perf_counter() - start_bench


INFO:openai._base_client:Retrying request to /chat/completions in 0.471019 seconds



FASE 6: Multi-model Benchmark
Avvio benchmark multi-modello...
  Calling Qwen2.5-7B (qwen2.5-7b-instruct) ... 

INFO:openai._base_client:Retrying request to /chat/completions in 0.782280 seconds


FAILED: Connection error.


In [41]:
# Tabella comparativa FairMind vs tutti i modelli LLM
rows = []
for model_name, res in llm_results.items():
    if res['effects'] is None:
        continue
    for ek in ['TV', 'TE', 'SE', 'DE', 'IE']:
        gt = manual_effects.get(ek, 0)
        pred = res['effects'].get(ek, 0)
        abs_err = abs(gt - pred)
        rel_err = abs_err / abs(gt) * 100 if abs(gt) > 1e-9 else float('nan')
        rows.append({
            'model': model_name,
            'effect': ek,
            'fairmind': round(gt, 6),
            'llm': round(pred, 6),
            'abs_error': round(abs_err, 6),
            'rel_error_%': round(rel_err, 2) if not np.isnan(rel_err) else float('nan'),
        })

df_discr = pd.DataFrame(rows)
print("\nDiscrepanze FairMind vs LLM:")
print(df_discr.to_string(index=False))



Discrepanze FairMind vs LLM:
Empty DataFrame
Columns: []
Index: []


## 9. Fase 7 — Generazione Report HTML

Salva tutti i risultati in un report HTML completo con tabelle, metriche e timing.


In [42]:
print("\n" + "=" * 60)
print("FASE 7: Generazione Report")
print("=" * 60)

llm_compact = {}
for model_name, res in llm_results.items():
    if res['effects'] is not None:
        llm_compact[model_name] = res['effects']

timing = {
    'discovery_seconds': round(discovery_time, 4),
    'fairmind_manual_seconds': round(fm_time, 4),
    'fairmind_learned_seconds': round(learned_time, 4),
    'intersectional_seconds': round(inter_time, 4),
    'benchmark_seconds': round(bench_time, 4),
}

report_path = generate_html_report(
    dataset_name=CONFIG['dataset_name'],
    config=CONFIG,
    manual_effects=manual_effects,
    learned_effects=learned_effects,
    llm_results=llm_compact,
    intersectional_results=inter_effects,
    discrepancies=df_discr,
    similarity_metrics=sim,
    timing=timing,
    output_path="pipeline_benchmark_results/multimodal_report.html",
)
print(f"\nReport generato: {report_path}")



FASE 7: Generazione Report
Report salvato: pipeline_benchmark_results/multimodal_report.html

Report generato: pipeline_benchmark_results/multimodal_report.html


## 10. Salvataggio risultati

Salva tutti i risultati in un JSON strutturato per analisi successive.


In [ ]:
import os, datetime
os.makedirs("pipeline_benchmark_results", exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"pipeline_benchmark_results/multimodal_{ts}.json"

out = {
    'dataset': CONFIG['dataset_name'],
    'config': {k: v for k, v in CONFIG.items() if k != 'csv_path'},
    'graph_similarity': sim,
    'fairmind_manual': manual_effects,
    'fairmind_learned': learned_effects,
    'llm_results': {k: {'effects': v['effects'], 'usage': v['usage'], 'time': v['time']}
                     for k, v in llm_results.items()},
    'intersectional': inter_effects,
    'discrepancies': df_discr.to_dict(orient='records') if len(df_discr) > 0 else [],
    'timing': timing,
}

with open(fname, 'w') as f:
    json.dump(out, f, indent=2)
print(f"Salvato: {fname}")


Salvato: pipeline_benchmark_results/multimodal_20260714_094808.json
